# examples-seen-step-axis — ex1: compute examples_seen = step * batch_size as wandb x-axis

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `examples-seen-step-axis`. Running the final beacon cell reports progress against the `Trainer: examples-seen step axis` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Trainer: examples-seen step axis` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`examples-seen-step-axis`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "examples-seen-step-axis"
DD_SUBTOPIC = "Trainer: examples-seen step axis"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `examples_seen = step * batch_size` — quick refresher

When comparing training runs that use DIFFERENT batch sizes, plotting loss vs `step` is misleading — a run with batch_size=64 takes half the steps of a batch_size=32 run to see the same amount of data. The fair x-axis is the number of EXAMPLES the model has seen:

```
examples_seen = step * batch_size
wandb.log({'loss': loss.item(), 'examples_seen': examples_seen})
```

Then in the wandb UI you set the x-axis to `examples_seen` and curves from different batch sizes overlay correctly.

**When step IS the right axis.** If you're comparing two runs with the SAME batch size, plotting against `step` is fine — and avoids the multiplication. The distinction matters when batch size varies.

**Partial last batch caveat.** `step * batch_size` slightly OVER-counts when the last batch of an epoch was partial. For most training graphs the error is < 1% and irrelevant; if you need the exact count, accumulate `batch.shape[0]` per step.

### Exercise 1 — compute examples_seen = step * batch_size as wandb x-axis

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply `examples_seen = step * batch_size` inside a training loop so logging the same number of examples processed produces overlapping curves across different batch sizes.
> Keywords: examples-seen, wandb-axis, logging, comparable-runs
> ```

**KCs targeted:** `examples-seen-equals-step-times-batch-size`, `examples-seen-as-fair-cross-batch-size-x-axis`

Implement `ex1_log_examples_seen(loss_history, batch_size)`. Convert a list of `(step, loss)` log tuples into the wandb-friendly `(examples_seen, loss)` format.

1. For each `(step, loss)` in `loss_history`, compute `examples_seen = step * batch_size`.
2. Return a new list of `(examples_seen, loss)` tuples in the same order.

Inputs:
- `loss_history`: list of `(step, loss)` tuples — `step` is 1-based int, `loss` is float.
- `batch_size`: int — examples per batch.

Output: list of `(examples_seen, loss)` tuples.

Then implement `ex1_runs_match_at_examples_seen(history_a, history_b, batch_a, batch_b)`. The PAYOFF of this transformation: two runs with DIFFERENT batch sizes that have seen the same number of examples should align on the x-axis even though their step counts differ.

Return `True` if the maximum `examples_seen` value matches across the two converted histories (within `batch_size` tolerance — partial last batches don't count exactly), `False` otherwise.

In [ ]:
def ex1_log_examples_seen(loss_history: list, batch_size: int) -> list:
    """Convert [(step, loss)] -> [(examples_seen, loss)] where examples_seen = step * batch_size."""
    raise NotImplementedError()


def ex1_runs_match_at_examples_seen(history_a: list, history_b: list,
                                    batch_a: int, batch_b: int) -> bool:
    """Do the two runs cover ~the same examples_seen range?"""
    raise NotImplementedError()


def _test_ex1():
    # === Basic conversion ===
    history = [(1, 1.0), (2, 0.9), (3, 0.8), (4, 0.7)]
    out = ex1_log_examples_seen(history, batch_size=32)
    expected = [(32, 1.0), (64, 0.9), (96, 0.8), (128, 0.7)]
    assert out == expected, f'expected {expected}, got {out}'

    # === Loss values preserved exactly (no rounding) ===
    history_f = [(1, 0.123456789), (2, 0.987654321)]
    out_f = ex1_log_examples_seen(history_f, batch_size=16)
    assert out_f[0][1] == 0.123456789 and out_f[1][1] == 0.987654321, (
        f'loss values must be preserved exactly; got {out_f}'
    )

    # === Type check: examples_seen is int, loss is float ===
    for ex_seen, l in out:
        assert isinstance(ex_seen, int), f'examples_seen must be int (step*batch_size); got {type(ex_seen)}'
        assert isinstance(l, float), f'loss must remain float; got {type(l)}'

    # === Order preserved ===
    history_shuffled = [(5, 0.5), (1, 1.0), (3, 0.8)]
    out_shuffled = ex1_log_examples_seen(history_shuffled, batch_size=10)
    assert out_shuffled == [(50, 0.5), (10, 1.0), (30, 0.8)], (
        f'order must match input; got {out_shuffled}'
    )

    # === Empty history ===
    assert ex1_log_examples_seen([], batch_size=64) == []

    # === Matching-runs payoff ===
    # Run A: batch_size=32, 10 batches → max examples_seen = 320.
    # Run B: batch_size=64,  5 batches → max examples_seen = 320.
    # Same examples seen even though step counts differ.
    hist_a = [(i, 1.0 / i) for i in range(1, 11)]
    hist_b = [(i, 1.0 / i) for i in range(1, 6)]
    assert ex1_runs_match_at_examples_seen(hist_a, hist_b, batch_a=32, batch_b=64) is True, (
        'run A (10 steps * 32) and run B (5 steps * 64) both reach 320 examples — should match'
    )

    # Mismatched runs.
    hist_c = [(i, 1.0 / i) for i in range(1, 11)]   # 10 steps
    hist_d = [(i, 1.0 / i) for i in range(1, 11)]   # 10 steps
    assert ex1_runs_match_at_examples_seen(hist_c, hist_d, batch_a=32, batch_b=64) is False, (
        'run C (10 * 32 = 320) and run D (10 * 64 = 640) should NOT match — different example counts'
    )

    # === Realistic scale ===
    big_history = [(s, 1.0 / (s + 1)) for s in range(1, 1001)]
    big_out = ex1_log_examples_seen(big_history, batch_size=128)
    assert len(big_out) == 1000
    assert big_out[-1][0] == 1000 * 128, f'last examples_seen wrong: {big_out[-1][0]}'
    assert big_out[0][0] == 128, f'first examples_seen wrong: {big_out[0][0]}'
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_log_examples_seen(loss_history, batch_size):
    return [(step * batch_size, loss) for step, loss in loss_history]


def ex1_runs_match_at_examples_seen(history_a, history_b, batch_a, batch_b):
    if not history_a or not history_b:
        return False
    max_a = max(s for s, _ in history_a) * batch_a
    max_b = max(s for s, _ in history_b) * batch_b
    tolerance = max(batch_a, batch_b)
    return abs(max_a - max_b) <= tolerance
```

**Why `step * batch_size` and not accumulating `x.shape[0]`.** For full-size batches they're identical. The accumulated form is exact when the last batch is partial (size < batch_size). For most training graphs the 1% error in the final batch doesn't matter — the simpler `step * batch_size` is what ARENA's wandb integration uses.

**When you'd reach for the accumulated form.** Curriculum learning, dataset streaming with variable batch sizes, or any setup where you NEED an exact example count (e.g. compute-budget comparisons across runs). For those, maintain `examples_seen += x.shape[0]` alongside `step += 1` in the loop body.

**Why this matters for cross-run comparison.** Compute-compare paper plots, scaling-law fits, and ablation grids all need a fair x-axis. Plotting against `step` privileges small-batch runs (they tick more steps per training-data epoch). Plotting against `examples_seen` makes runs of different batch sizes directly visually comparable — and that's exactly what wandb 'set x-axis' dropdown is for.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()